
---
## Section 3 - Exploratory Data Analysis (EDA)

Summary statistics, class balance, feature correlations, and distributions are examined to inform feature selection and model hyperparameters.


In [ ]:

# 3.1 Summary Statistics
summary = df[FEATURES + [TARGET]].describe().T[['mean','std','min','50%','max']]
summary.columns = ['Mean', 'Std Dev', 'Min', 'Median', 'Max']
print("=" * 70)
print("  FEATURE SUMMARY STATISTICS (full dataset, n=1265)")
print("=" * 70)
print(summary.round(4).to_string())


NameError: name 'df' is not defined

In [ ]:

# 3.2 Target Distribution & Season Balance
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#0d1117')

# A) Global class distribution
bars = axes[0].bar(['Team B Wins (y=0)', 'Team A Wins (y=1)'],
                   [y_train.value_counts()[0], y_train.value_counts()[1]],
                   color=[PALETTE[1], PALETTE[0]], edgecolor='white', linewidth=0.5)
axes[0].set_title('Target Distribution (Training Set)', fontweight='bold')
axes[0].set_ylabel('Match Count')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2, h+4, f'{h}\n({100*h/len(y_train):.1f}%)',
                 ha='center', va='bottom', fontsize=10)

# B) Win rate per season
season_wr = df.groupby('season')['team_a_won'].mean()
axes[1].bar([f'D{s-2022}\n({s})' for s in season_wr.index],
            season_wr.values, color=PALETTE[:3], edgecolor='white', linewidth=0.5)
axes[1].axhline(0.5, color='white', ls='--', alpha=0.7, label='50% baseline')
axes[1].set_ylim(0.4, 0.65)
axes[1].set_title('Team A Win Rate by Season', fontweight='bold')
axes[1].set_ylabel('Win Rate')
axes[1].legend()

# C) Stage stakes distribution
stake_counts = df['stage_stakes'].value_counts().sort_index()
stake_labels = {1: 'League\nPlay', 2: 'Group/\nKnockout', 3: 'Playoff/\nFinals'}
axes[2].bar([stake_labels[k] for k in stake_counts.index],
            stake_counts.values, color=[PALETTE[0], PALETTE[3], PALETTE[2]],
            edgecolor='white', linewidth=0.5)
axes[2].set_title('Match Distribution by Stage Stakes', fontweight='bold')
axes[2].set_ylabel('Match Count')

for ax in axes:
    ax.grid(axis='y', alpha=0.4)
    ax.set_facecolor('#161b22')

plt.suptitle('EDA - Target & Dataset Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print(f"\nEDA Insight: The training dataset is near-balanced ({100*y_train.mean():.1f}% / {100*(1-y_train.mean()):.1f}%)")
print("   Team A has a slight structural advantage - in VCT brackets, Team A is typically the higher seed.")


In [ ]:

# 3.3 Feature Correlation Heatmap
corr_matrix = df[FEATURES + [TARGET]].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.patch.set_facecolor('#0d1117')

# Full correlation heatmap
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_matrix, mask=mask, ax=axes[0], annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot_kws={'size': 7}, linewidths=0.3,
            cbar_kws={'shrink': 0.8})
axes[0].set_title('Full Feature Correlation Matrix', fontweight='bold', fontsize=12)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)
axes[0].set_facecolor('#161b22')

# Feature–target correlation (bar chart)
target_corr = corr_matrix[TARGET].drop(TARGET).sort_values()
colors = [PALETTE[1] if v < 0 else PALETTE[0] for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors, edgecolor='none')
axes[1].axvline(0, color='white', linewidth=0.8)
axes[1].set_title('Feature Correlation with Target (team_a_won)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Pearson Correlation Coefficient')
axes[1].set_facecolor('#161b22')

plt.tight_layout()
plt.show()
print("\nEDA Insight: Differential features (hist_win_rate_diff, map_win_pct_diff,")
print("   hist_rating_diff) are the strongest predictors, confirming the intuition")
print("   that relative strength - not absolute performance - drives match outcomes.")
print("   Context features (stage_stakes, is_elimination_match) show near-zero")
print("   correlation, suggesting these features capture variance unexplained by")
print("   pure historical performance.")


In [ ]:

# 3.4 Feature Distributions by Outcome
top_features = ['hist_win_rate_diff', 'hist_rating_diff', 'map_win_pct_diff',
                'ta_hist_win_rate', 'tb_hist_win_rate', 'stage_stakes']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.patch.set_facecolor('#0d1117')

for ax, feat in zip(axes.flat, top_features):
    for label, color, lbl in [(1, PALETTE[0], 'Team A Wins'), (0, PALETTE[1], 'Team B Wins')]:
        data = df.loc[df[TARGET] == label, feat]
        ax.hist(data, bins=30, alpha=0.6, color=color, label=lbl, edgecolor='none', density=True)
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.set_facecolor('#161b22')
    ax.grid(alpha=0.3)

plt.suptitle('Feature Distributions by Match Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nEDA Insight: Differential features (hist_win_rate_diff, hist_rating_diff)")
print("   show clear separation between winning and losing teams, validating their")
print("   predictive utility. Individual win-rate distributions overlap considerably,")
print("   reinforcing the value of pairwise differential features.")
